# Nonlinear Model Comparison

This notebook compares the selected Ridge baseline with Random Forest models using the same prepared player-season data, chronological splits, and selected features from the Ridge notebook.

The aim is to choose a stronger model for 2025/26 valuation-gap rankings, while keeping the 2024/25 season as the final held-out test.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
model_data = pd.read_csv(
    "../data/interim/labelled_player_seasons.csv.gz",
    parse_dates=["season_end_date", "valuation_date", "date_of_birth"],
)

scoring_data = pd.read_csv(
    "../data/interim/scoring_2025_26.csv.gz",
    parse_dates=["season_end_date", "date_of_birth"],
)

model_data.shape, scoring_data.shape

In [ ]:
def add_engineered_features(data):
    result = data.copy()

    result["age_distance_squared"] = (
        result["age_at_season_end"] - 25
    ) ** 2

    return result


model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

In [ ]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

In [ ]:
target = "market_value_in_eur"

selected_numeric_features = [
    "appearances",
    "minutes_played",
    "goals",
    "assists",
    "yellow_cards",
    "red_cards",
    "clubs_played_for",
    "height_in_cm",
    "age_at_season_end",
    "age_distance_squared",
]

selected_categorical_features = [
    "position",
    "sub_position",
    "foot",
]

selected_features = selected_numeric_features + selected_categorical_features

In [ ]:
def evaluate_predictions(actual, predicted):
    return {
        "mae": mean_absolute_error(actual, predicted),
        "rmse": root_mean_squared_error(actual, predicted),
        "r2": r2_score(actual, predicted),
    }

In [ ]:
def make_ridge_model(alpha=100):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, selected_numeric_features),
        ("categorical", categorical_transformer, selected_categorical_features),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha)),
    ])


def make_random_forest_model():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, selected_numeric_features),
        ("categorical", categorical_transformer, selected_categorical_features),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

In [ ]:
def walk_forward_evaluate(model_builder):
    rows = []

    for validation_season in range(2019, 2024):
        fold_train = model_data.loc[
            model_data["season"].between(2016, validation_season - 1)
        ]

        fold_validation = model_data.loc[
            model_data["season"].eq(validation_season)
        ]

        model = model_builder()

        model.fit(
            fold_train[selected_features],
            fold_train[target],
        )

        predictions = np.clip(
            model.predict(fold_validation[selected_features]),
            0,
            None,
        )

        rows.append({
            "validation_season": validation_season,
            **evaluate_predictions(fold_validation[target], predictions),
        })

    return pd.DataFrame(rows)

## Baseline nonlinear comparison

Compare the frozen Ridge baseline against a plain Random Forest using walk-forward validation.

In [ ]:
ridge_cv = walk_forward_evaluate(make_ridge_model)
forest_cv = walk_forward_evaluate(make_random_forest_model)

model_comparison = pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
}).T

model_comparison

In [ ]:
fold_comparison = ridge_cv.merge(
    forest_cv,
    on="validation_season",
    suffixes=("_ridge", "_forest"),
)

fold_comparison["mae_improvement"] = (
    fold_comparison["mae_ridge"] - fold_comparison["mae_forest"]
)

fold_comparison

## Final Random Forest test evaluation

Random Forest was selected using walk-forward validation only. We now evaluate it once on the untouched 2024/25 test season.

In [ ]:
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True,
)

final_forest_model = make_random_forest_model()

final_forest_model.fit(
    development_data[selected_features],
    development_data[target],
)

test_predictions = np.clip(
    final_forest_model.predict(test_data[selected_features]),
    0,
    None,
)

pd.Series(
    evaluate_predictions(test_data[target], test_predictions),
    name="2024/25 final test",
)

## Random Forest residual analysis

Inspect the model's biggest misses to see whether the nonlinear model reduces the Ridge baseline's tendency to compress elite players toward the average.

In [ ]:
test_results = test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "market_value_in_eur",
    ]
].copy()

test_results["predicted_value"] = test_predictions
test_results["residual"] = (
    test_results["market_value_in_eur"] - test_results["predicted_value"]
)
test_results["absolute_error"] = test_results["residual"].abs()

test_results.sort_values("absolute_error", ascending=False).head(15)

## Add historical features

The first Random Forest only sees the current season. We now add previous-season performance and previous market value so the model has memory of player reputation and recent history.

In [ ]:
history_columns = [
    "player_id",
    "season",
    "minutes_played",
    "goals",
    "assists",
    "appearances",
    "market_value_in_eur",
]

player_history = model_data[history_columns].sort_values(
    ["player_id", "season"]
).copy()

for column in [
    "minutes_played",
    "goals",
    "assists",
    "appearances",
    "market_value_in_eur",
]:
    player_history[f"previous_{column}"] = (
        player_history.groupby("player_id")[column].shift(1)
    )

history_features = player_history[
    [
        "player_id",
        "season",
        "previous_minutes_played",
        "previous_goals",
        "previous_assists",
        "previous_appearances",
        "previous_market_value_in_eur",
    ]
]

model_data_with_history = model_data.merge(
    history_features,
    on=["player_id", "season"],
    how="left",
)

model_data_with_history[
    [
        "player_name",
        "season",
        "minutes_played",
        "previous_minutes_played",
        "market_value_in_eur",
        "previous_market_value_in_eur",
    ]
].head(10)

In [ ]:
history_numeric_features = selected_numeric_features + [
    "previous_minutes_played",
    "previous_goals",
    "previous_assists",
    "previous_appearances",
    "previous_market_value_in_eur",
]

history_features = history_numeric_features + selected_categorical_features

In [ ]:
def make_history_random_forest_model():
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, history_numeric_features),
        ("categorical", categorical_transformer, selected_categorical_features),
    ])

    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=300,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        )),
    ])

In [ ]:
def walk_forward_evaluate_history(model_builder):
    rows = []

    for validation_season in range(2019, 2024):
        fold_train = model_data_with_history.loc[
            model_data_with_history["season"].between(2016, validation_season - 1)
        ]

        fold_validation = model_data_with_history.loc[
            model_data_with_history["season"].eq(validation_season)
        ]

        model = model_builder()

        model.fit(
            fold_train[history_features],
            fold_train[target],
        )

        predictions = np.clip(
            model.predict(fold_validation[history_features]),
            0,
            None,
        )

        rows.append({
            "validation_season": validation_season,
            **evaluate_predictions(fold_validation[target], predictions),
        })

    return pd.DataFrame(rows)

## History model comparison

Compare the plain Random Forest against the history-aware Random Forest using the same walk-forward folds.

In [ ]:
history_forest_cv = walk_forward_evaluate_history(make_history_random_forest_model)

pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
    "history_random_forest": history_forest_cv[["mae", "rmse", "r2"]].mean(),
}).T

In [ ]:
history_fold_comparison = forest_cv.merge(
    history_forest_cv,
    on="validation_season",
    suffixes=("_forest", "_history_forest"),
)

history_fold_comparison["mae_improvement"] = (
    history_fold_comparison["mae_forest"]
    - history_fold_comparison["mae_history_forest"]
)

history_fold_comparison

## Final history Random Forest test evaluation

The history Random Forest was selected using walk-forward validation. We now evaluate it once on the untouched 2024/25 test season.

In [ ]:
development_data_with_history = model_data_with_history.loc[
    model_data_with_history["season"].between(2016, 2023)
].copy()

test_data_with_history = model_data_with_history.loc[
    model_data_with_history["season"].eq(2024)
].copy()

final_history_forest_model = make_history_random_forest_model()

final_history_forest_model.fit(
    development_data_with_history[history_features],
    development_data_with_history[target],
)

history_test_predictions = np.clip(
    final_history_forest_model.predict(test_data_with_history[history_features]),
    0,
    None,
)

pd.Series(
    evaluate_predictions(test_data_with_history[target], history_test_predictions),
    name="2024/25 final test",
)

In [ ]:
history_test_results = test_data_with_history[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "previous_minutes_played",
        "market_value_in_eur",
        "previous_market_value_in_eur",
    ]
].copy()

history_test_results["predicted_value"] = history_test_predictions
history_test_results["residual"] = (
    history_test_results["market_value_in_eur"]
    - history_test_results["predicted_value"]
)
history_test_results["absolute_error"] = history_test_results["residual"].abs()

history_test_results.sort_values("absolute_error", ascending=False).head(15)

In [ ]:
history_test_results["has_previous_market_value"] = (
    history_test_results["previous_market_value_in_eur"].notna()
)

history_test_results.groupby("has_previous_market_value").agg(
    players=("player_id", "count"),
    actual_mean=("market_value_in_eur", "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

## Score 2025/26 players

Apply the selected history Random Forest model to the 2025/26 scoring population. The prediction is an expected Transfermarkt-style value; the valuation gap is the model prediction minus the current Transfermarkt value.

In [ ]:
latest_history = model_data_with_history.loc[
    model_data_with_history["season"].eq(2024),
    [
        "player_id",
        "minutes_played",
        "goals",
        "assists",
        "appearances",
        "market_value_in_eur",
    ],
].rename(columns={
    "minutes_played": "previous_minutes_played",
    "goals": "previous_goals",
    "assists": "previous_assists",
    "appearances": "previous_appearances",
    "market_value_in_eur": "previous_market_value_in_eur",
})

scoring_data_with_history = scoring_data.merge(
    latest_history,
    on="player_id",
    how="left",
)

scoring_data_with_history[
    [
        "player_name",
        "season",
        "minutes_played",
        "previous_minutes_played",
        "previous_market_value_in_eur",
    ]
].head(10)

In [ ]:
players = pd.read_csv("../data/raw/players.csv.gz")

current_player_values = players[
    [
        "player_id",
        "market_value_in_eur",
        "current_club_name",
        "current_club_domestic_competition_id",
    ]
].rename(columns={
    "market_value_in_eur": "current_market_value_in_eur",
})

scoring_data_with_history = scoring_data_with_history.drop(
    columns=[
        column for column in [
            "current_market_value_in_eur",
            "current_club_name",
            "current_club_domestic_competition_id",
        ]
        if column in scoring_data_with_history.columns
    ]
).merge(
    current_player_values,
    on="player_id",
    how="left",
)

scoring_data_with_history[
    [
        "player_name",
        "current_club_name",
        "current_club_domestic_competition_id",
        "minutes_played",
        "previous_market_value_in_eur",
        "current_market_value_in_eur",
    ]
].head(10)

In [ ]:
scoring_predictions = np.clip(
    final_history_forest_model.predict(scoring_data_with_history[history_features]),
    0,
    None,
)

scoring_results = scoring_data_with_history[
    [
        "season",
        "player_id",
        "player_name",
        "current_club_name",
        "current_club_domestic_competition_id",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "previous_minutes_played",
        "current_market_value_in_eur",
        "previous_market_value_in_eur",
    ]
].copy()
scoring_results["predicted_value"] = scoring_predictions
scoring_results["valuation_gap"] = (
    scoring_results["predicted_value"]
    - scoring_results["current_market_value_in_eur"]
)
scoring_results["absolute_gap"] = scoring_results["valuation_gap"].abs()

scoring_results["has_previous_market_value"] = (
    scoring_results["previous_market_value_in_eur"].notna()
)

scoring_results["minutes_band"] = pd.cut(
    scoring_results["minutes_played"],
    bins=[-1, 449, 899, np.inf],
    labels=["low_minutes", "medium_minutes", "regular_minutes"],
)

scoring_results[
    [
        "player_name",
        "current_club_name",
        "position",
        "minutes_played",
        "current_market_value_in_eur",
        "predicted_value",
        "valuation_gap",
        "has_previous_market_value",
        "minutes_band",
    ]
].head()

## 2025/26 valuation-gap rankings

Filter to current Premier League players with current Transfermarkt values, then rank regular-minute players by valuation gap.

In [ ]:
ranking_pool = scoring_results.loc[
    scoring_results["current_market_value_in_eur"].notna()
    & scoring_results["current_club_domestic_competition_id"].eq("GB1")
].copy()

undervalued_players = (
    ranking_pool.loc[
        ranking_pool["minutes_band"].eq("regular_minutes")
    ]
    .sort_values("valuation_gap", ascending=False)
    [
        [
            "player_name",
            "current_club_name",
            "position",
            "sub_position",
            "age_at_season_end",
            "minutes_played",
            "current_market_value_in_eur",
            "predicted_value",
            "valuation_gap",
            "has_previous_market_value",
        ]
    ]
    .head(20)
)

undervalued_players

In [ ]:
overvalued_players = (
    ranking_pool.loc[
        ranking_pool["minutes_band"].eq("regular_minutes")
    ]
    .sort_values("valuation_gap", ascending=True)
    [
        [
            "player_name",
            "current_club_name",
            "position",
            "sub_position",
            "age_at_season_end",
            "minutes_played",
            "current_market_value_in_eur",
            "predicted_value",
            "valuation_gap",
            "has_previous_market_value",
        ]
    ]
    .head(20)
)

overvalued_players

## High-confidence bargain views

Focus on regular-minute players with previous market values. This removes many new-arrival cases where the model has less historical context.

In [ ]:
high_confidence_pool = ranking_pool.loc[
    ranking_pool["minutes_band"].eq("regular_minutes")
    & ranking_pool["has_previous_market_value"]
].copy()

high_confidence_undervalued = (
    high_confidence_pool
    .sort_values("valuation_gap", ascending=False)
    [
        [
            "player_name",
            "current_club_name",
            "position",
            "sub_position",
            "age_at_season_end",
            "minutes_played",
            "previous_market_value_in_eur",
            "current_market_value_in_eur",
            "predicted_value",
            "valuation_gap",
        ]
    ]
    .head(20)
)

high_confidence_undervalued

In [ ]:
recruitment_bargains = (
    high_confidence_pool.loc[
        high_confidence_pool["age_at_season_end"].lt(27)
        & high_confidence_pool["current_market_value_in_eur"].le(50_000_000)
    ]
    .sort_values("valuation_gap", ascending=False)
    [
        [
            "player_name",
            "current_club_name",
            "position",
            "sub_position",
            "age_at_season_end",
            "minutes_played",
            "previous_market_value_in_eur",
            "current_market_value_in_eur",
            "predicted_value",
            "valuation_gap",
        ]
    ]
    .head(20)
)

recruitment_bargains

## Experiment log

- Random Forest beat the selected Ridge baseline on walk-forward validation and final 2024/25 test performance.
- Adding previous-season performance and previous market value produced the largest improvement, reducing final test MAE from about EUR 9.46m to EUR 5.80m.
- The history model is strongest for players with previous Premier League market values and weaker for new arrivals or breakout players without prior PL history.
- 2025/26 scoring produced three useful ranking views: candidate undervalued players, candidate overvalued players, and a recruitment-style bargain shortlist.